# Algorithms in ySights

This tutorial covers the analytical algorithms available in ySights for understanding simulation dynamics.

## What You'll Learn

- Profile similarity analysis
- Recommendation system metrics
- Topic lifecycle analysis
- Moderation and forum session summaries

---

In [ ]:
from pathlib import Path

from ysights import YDataHandler
from ysights.algorithms import (
    profile_topics_similarity,
    visibility_paradox,
    user_visibility_vs_neighbors,
    visibility_paradox_population_size_null,
    engagement_momentum,
    personalization_balance_score,
)
from ysights.algorithms.topics import topic_spread, adoption_rate, peak_engagement_time
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
# Initialize data handler and get network
from pathlib import Path


def resolve_example_db():
    candidates = [
        Path("ysocial_db.db"),
        Path("../notebooks/ysocial_db.db"),
        Path("../../notebooks/ysocial_db.db"),
        Path("docs/notebooks/ysocial_db.db"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate.resolve())
    return "ysocial_db.db"

db_path = resolve_example_db()
ydh = YDataHandler(db_path)
network = ydh.social_network()

## 1. Profile Similarity Analysis

Analyzes how similar users' interest profiles are across the network.

In [ ]:
similarity_scores = profile_topics_similarity(ydh, network)
similarity_values = list(similarity_scores.values())

print(f"Computed {len(similarity_scores)} similarity scores")
print(f"\nSimilarity Statistics:")
print(f"  Mean: {np.mean(similarity_values):.4f}")
print(f"  Median: {np.median(similarity_values):.4f}")
print(f"  Std Dev: {np.std(similarity_values):.4f}")
print(f"  Min: {min(similarity_values):.4f}")
print(f"  Max: {max(similarity_values):.4f}")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(similarity_values, bins=50, edgecolor='black', alpha=0.7, color='coral')
plt.xlabel('Similarity Score', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.title('Profile Similarity Distribution', fontsize=13, fontweight='bold')
plt.axvline(np.mean(similarity_values), color='red', linestyle='--', label=f'Mean: {np.mean(similarity_values):.3f}')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(similarity_values, vert=True)
plt.ylabel('Similarity Score', fontsize=11)
plt.title('Profile Similarity Box Plot', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 2. Recommendation System Metrics

### Engagement Momentum

Measures how consistently users engage with recommended content over time.

In [ ]:
momentum = engagement_momentum(ydh, time_window_rounds=24)

print("Engagement Momentum Analysis:")
print(f"  Users analyzed: {len(momentum)}")
print(f"  Average momentum: {np.mean(list(momentum.values())):.4f}")
print(f"  Median momentum: {np.median(list(momentum.values())):.4f}")

top_momentum = sorted(momentum.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 Users by Engagement Momentum:")
for i, (user, score) in enumerate(top_momentum, 1):
    print(f"  {i}. User {user}: {score:.4f}")

### Personalization Balance Score

Measures how well the recommendation system balances exploration vs. exploitation.

In [ ]:
balance_scores = personalization_balance_score(ydh)

print("Personalization Balance Analysis:")
print(f"  Users analyzed: {len(balance_scores)}")
print(f"  Average balance: {np.mean(list(balance_scores.values())):.4f}")
print(f"  Median balance: {np.median(list(balance_scores.values())):.4f}")
print("\nInterpretation:")
print("  Score → 0: Heavy exploitation (narrow recommendations)")
print("  Score → 1: Heavy exploration (diverse recommendations)")

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(list(balance_scores.values()), bins=30, edgecolor='black', alpha=0.7, color='seagreen')
plt.xlabel('Personalization Balance Score', fontsize=12)
plt.ylabel('Number of Users', fontsize=12)
plt.title('Distribution of Personalization Balance', fontsize=14, fontweight='bold')
plt.axvline(0.5, color='red', linestyle='--', linewidth=2, label='Perfect Balance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Topic Lifecycle Analysis

These helpers are thin wrappers around the implemented topic lifecycle analysis in `YDataHandler`.

In [ ]:
lifecycles = topic_spread(ydh)
adoption_rates = adoption_rate(ydh)
peak_periods = peak_engagement_time(ydh)

topic_ids = list(lifecycles.keys())
print(f"Topics analyzed: {len(topic_ids)}")

for topic_id in topic_ids[:5]:
    lifecycle = lifecycles[topic_id]
    print(
        f"  Topic {topic_id}: posts={lifecycle['post_count']}, "
        f"authors={lifecycle['author_count']}, "
        f"peak={lifecycle['peak_period']}, "
        f"adoption={adoption_rates[topic_id]:.3f}"
    )

if topic_ids:
    topic_id = topic_ids[0]
    print(f"\nTimeline preview for topic {topic_id}:")
    print(lifecycles[topic_id]["timeline"].head().to_string(index=False))

## 4. Operational and Moderation Summaries

These helpers are useful when you want a compact dataset overview before running a deeper analysis.

In [ ]:
summary_report = ydh.summary_report()
summary_frame = ydh.summary_frame()
moderation_summary = ydh.moderation_summary()
forum_sessions = ydh.forum_session_summaries()
cache_info = ydh.analysis_cache_info()
recommended_indexes = ydh.recommended_indexes()
benchmark = ydh.benchmark_analytics(iterations=1)

print("Summary Report (selected keys):")
for key in ["post_count", "thread_count", "report_count", "forum_session_count", "moderated_posts"]:
    if key in summary_report:
        print(f"  {key}: {summary_report[key]}")

print("\nSummary Frame Preview:")
print(summary_frame.head().to_string(index=False))

print("\nModeration Summary:")
print(moderation_summary)

print("\nForum Session Summaries:")
print(forum_sessions)

print("\nCache Diagnostics:")
print(cache_info)

print("\nRecommended Indexes:")
print(recommended_indexes)

print("\nBenchmark Metrics:")
print(list(benchmark["metrics"].keys()))

## Summary

In this tutorial, you learned:

✓ How to measure profile similarity across the network
✓ Computing recommendation system metrics (engagement momentum, personalization balance)  
✓ Analyzing topic spread, adoption, and peak engagement
✓ Reviewing moderation, forum session, and summary diagnostics

## Next Steps

- **Visualization Tutorial**: Create publication-ready visualizations using ySights' viz module